# 01. 세션 하나 열어보기

새 데이터를 받으면 제일 먼저 돌리는 노트북.
1. 세션 안에 어떤 CSV가 있는지, 실제 주파수가 몇 Hz인지 확인 → `docs/spec_data_format.md` 채우기
2. 걸음 신호와 검출된 걸음을 눈으로 확인
3. 헤딩, 궤적 확인

`SESSION` 만 바꿔가며 실행하세요. 파라미터를 바꿔보고 싶으면 `params.with_(...)` 셀을 고치면 됩니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from indoornav.io import load_session, describe
from indoornav.pdr import PDRParams, run_pdr
from indoornav.eval import load_ground_truth, evaluate
from indoornav.eval.plots import plot_steps, plot_heading, plot_trajectory

SESSION = "20261012_S1_normal_01"          # ← 바꿔가며 실행
path = ROOT / "data" / SESSION

## 1. 파일 구성 / 실제 주파수

In [ ]:
describe(path)

## 2. 불러오기 (100 Hz 공통 시간축)

In [ ]:
s = load_session(path)
print(f"{s.name}: {s.duration:.1f} s, iPhone 걸음 수 = {s.iphone_steps}")
print("중력 크기 중앙값:", np.median(np.linalg.norm(s.gravity, axis=1)).round(3), "m/s²  (9.81 근처여야 정상)")
print("정지 구간 가속도 크기:", np.linalg.norm(s.acc_user[: int(2 * s.fs)], axis=1).mean().round(3), "m/s²  (0 근처여야 정상)")
s.annotations

## 3. PDR 실행 (기본 파라미터)

In [ ]:
params = PDRParams.from_toml(ROOT / "configs/pdr_default.toml")
# 실험: 일부 파라미터만 바꿔보기
# params = params.with_(**{"step.min_height": 0.3, "preprocess.lpf_cutoff_hz": 2.5})

r = run_pdr(s, params)
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_steps(s, r, axes[0]); plot_heading(s, r, axes[1]); fig.tight_layout()
plot_trajectory(r);

## 4. 정답과 비교

In [ ]:
gt = load_ground_truth(ROOT / "docs/sessions.csv", ROOT / "docs/routes.csv")[SESSION]
evaluate(r, gt, s.iphone_steps)

## 5. 걸음 하나 확대 (기준값 감 잡기)
느린 걸음에서 봉우리 높이가 `min_height` 근처인지 확인하세요.

In [ ]:
t0, t1 = 5, 10          # 확대할 구간 [s]
m = (s.t >= t0) & (s.t <= t1)
plt.figure(figsize=(12, 3))
plt.plot(s.t[m], r.signal[m])
idx = r.steps["idx"].to_numpy(); idx = idx[(s.t[idx] >= t0) & (s.t[idx] <= t1)]
plt.plot(s.t[idx], r.signal[idx], "rv")
plt.axhline(params.step.min_height, color="gray", ls="--"); plt.grid(alpha=.3)